In [ ]:
from pyspark.sql.session import SparkSession
spark = (SparkSession.builder
                .master("local")
                .appName("withColumnFunctionUsage")
                .getOrCreate())

help(SparkSession)

In [ ]:
people = spark.createDataFrame([
            {"deptId": 1, "age": 40, "name": "Hyukjin Kwon", "gender": "M", "salary": 50},
            {"deptId": 1, "age": 50, "name": "Takuya Ueshin", "gender": "M", "salary": 100},
            {"deptId": 2, "age": 60, "name": "Xinrong Meng", "gender": "F", "salary": 150},
            {"deptId": 3, "age": 20, "name": "Haejoon Lee", "gender": "M", "salary": 200}
         ])

age_col = people.age
age_col

In [ ]:
department = spark.createDataFrame([
            {"id": 1, "name": "PySpark"},
            {"id": 2, "name": "ML"},
            {"id": 3, "name": "Spark SQL"}
         ])

people.filter(people.age > 30).show()

In [ ]:
people.filter(people.age > 30).join(
    department, people.deptId == department.id
)\
.groupBy(department.name, people.gender)\
.agg({"salary": "avg", "age": "max"}).show()

In [ ]:
department.select("name").show()
department.select(department.name).show()
department.select(department["name"]).show()

In [ ]:
df = spark.range(3)
# dir(df) lists all the attributes(columns & methods) available on the supplied object
# here it lists all the methods that are available on the DataFrame object
dir(df)

In [ ]:
# lists all the attributes (columns and methods) that starts with 'i' and present in DataFrame
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
from pyspark.sql.functions import *
# # add a new column named 'i_like_pancakes'
# it's not going to alter your existing df, instead returns a new df with the new column
df = df.withColumn('i_like_pancakes', lit(1))
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
# try to add an existed column 'inputFiles' -- no change/overwrites the existing column
df = df.withColumn('inputFiles', lit(3))
[attr for attr in dir(df) if attr[0] == 'i']

In [ ]:
# Dont include columns that are not valid python identifiers
df.withColumn('1', lit(4)).withColumn('name 1', lit(5)).show()

In [ ]:
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.show()
print(type(df.age))
print(type(df.select(df.age)))
df.select(df.age).show()

In [ ]:
# select columns based on indexes
df.select(df[0]).show()
df.select(df[1]).show()

In [ ]:
# select multiple string columns as index
df.select("name", "age").show()
df[df.age > 3].show()

In [ ]:
# agg --  Aggregate on the entire :class:`DataFrame` without groups
df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])

# duplicate keys in dict overwrites the older value. Hence if you need multiple aggregation on the same column use 
df.agg({"age": "max", "age": "avg"}).show()

from pyspark.sql.functions import *
df.agg(max("age"), avg("age"), min("age")).show()

In [ ]:
# alias(self, alias: str) -> 'DataFrame'
# Returns a new :class:`DataFrame` with an alias set.

df = spark.createDataFrame([(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])

df_as1 = df.alias("df_as1")
df_as2 = df.alias("df_as2")

df_as1.show()
df_as2.show()

# by default -- it does inner join
joined_df = df_as1.join(df_as2, df_as1.age == df_as2.age)
joined_df.show()

# sort(desc(col)) sorts the df in desc order on basis of specified column
joined_df.select("df_as2.age", "df_as1.name").sort(desc("df_as2.age")).show()

In [ ]:
# cache(self) -> 'DataFrame'
#  |      Persists the :class:`DataFrame` with the default storage level (`MEMORY_AND_DISK_DESER`).
df = spark.range(3)
df.cache()

In [ ]:
# Prints the (logical and physical) plans to the console for debugging purposes.
# Parameters
#     ----------
#     extended : bool, optional
#         default ``False``. If ``False``, prints only the physical plan.
#         When this is a string without specifying the ``mode``, it works as the mode is
#         specified.
#     mode : str, optional
#         specifies the expected output format of plans.
    
#         * ``simple``: Print only a physical plan.
#         * ``extended``: Print both logical and physical plans.
#         * ``codegen``: Print a physical plan and generated codes if they are available.
#         * ``cost``: Print a logical plan and statistics if they are available.
#         * ``formatted``: Split explain output into two sections: a physical plan outline                 and node details.


# by default -- only prints physical plans
df.explain()

In [ ]:
# `extended``: Print both logical and physical plans.
df.explain('extended')

In [ ]:
# checkpoint(self, eager: bool = True) -> 'DataFrame'
#  |      Returns a checkpointed version of this :class:`DataFrame`. Checkpointing can be used to
#  |      truncate the logical plan of this :class:`DataFrame`, which is especially useful in
#  |      iterative algorithms where the plan may grow exponentially. It will be saved to files
#  |      inside the checkpoint directory set with :meth:`SparkContext.setCheckpointDir`.
 
    
#      Parameters
#  |      ----------
#  |      eager : bool, optional, default True
#  |          Whether to checkpoint this :class:`DataFrame` immediately.


import tempfile
df = spark.createDataFrame([
            (14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])

from tempfile import TemporaryDirectory() as d:
    # all the logical plans will be stored here
    spark.sparkContext.setCheckpointDir("/tmp/bb")
    df.checkpoint(false)
    

In [ ]:
# coalesce(numPartitions) --> DataFrame
# Coalesce is mainly used to reduce the num of partitions. 
# Avoids shuffling (or minimises it)
# Faster, but can create skewed partitions

# repartition(numPartition, columns)
# Repartition can increase or decrease partitions
# Uses full shuffle of data
# Slower, but produces evenly balanced partitions

# When to use what

# Use repartition when:
#     You need more partitions
#     You need balanced data
#     You repartition by a column

# Use coalesce when:
#     You just want to reduce partitions quickly
#     Data skew is not a big concern
#     Before writing small number of output files

df = spark.range(10)

# reduce the num of partitions to 1
df.coalesce(1).rdd.getNumPartitions()

In [ ]:
#   collect(self) -> List[pyspark.sql.types.Row]
#  |      Returns all the records as a list of :class:`Row`.

df = spark.createDataFrame(
        [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.collect()

In [ ]:
# corr(col1, col2) -- Calculates the correlation of two columns of a :class:`DataFrame` as a double value.

# corr() in PySpark is used to measure how strongly two numeric columns are related to each other.

# What it actually does
# corr(col1, col2) computes the Pearson correlation coefficient between two columns.
# Value range: -1 to +1

# Meaning:
# +1 → perfect positive relationship
# -1 → perfect negative relationship
# 0 → no linear relationship

# It helps you answer questions like:
# Do two metrics move together?
#     Example: ad_spend vs revenue
# Is one variable a good predictor of another?
# Is there redundancy between features in ML?
# Should two columns be combined, removed, or transformed?

df = spark.createDataFrame([(1, 12), (10, 1), (19, 8)], ["c1", "c2"])
df.corr("c1", "c2")

In [ ]:
# count(self) -> int
#  |      Returns the number of rows in this :class:`DataFrame`.
df = spark.createDataFrame(
        [(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df.count()

In [ ]:
# createGlobalTempView(self, name: str) -> None
#  |      Creates a global temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary view is tied to this Spark application.
#  |      throws :class:`TempTableAlreadyExistsException`, if the view name already exists in the
#  |      catalog.

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createGlobalTempView("people")
df2 = spark.sql("select * from global_temp.people")

In [ ]:
# very good way to compare if 2 Dataframes are equivalent
# call df.collect() -- this returns list of Row() objects. Sort the list and compare
sorted(df2.collect()) == sorted(df.collect())

In [ ]:
# drop the GlobalTempView.
spark.catalog.dropGlobalTempView("people")

In [ ]:
# createOrReplaceGlobalTempView(self, name: str) -> None
#  |      Creates or replaces a global temporary view using the given name.
#  |      
#  |      The lifetime of this temporary view is tied to this Spark application.

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createGlobalTempView("people")

In [ ]:
# replace the global temp view 
df2 = df[df.age > 3]
df2.createOrReplaceGlobalTempView("people")

In [ ]:
df3 = spark.sql("select * from global_temp.people")
sorted(df2.collect()) == sorted(df3.collect())

In [ ]:
spark.catalog.dropGlobalTempView("people")

In [ ]:
#  createOrReplaceTempView(self, name: str) -> None
#  |      Creates or replaces a local temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary table is tied to the :class:`SparkSession`
#  |      that was used to create this :class:`DataFrame`.

df = spark.createDataFrame([(2, "Alice"), (5, "Bob")], schema=["age", "name"])
df.createOrReplaceTempView("people")

In [ ]:
# replace the local temp view 
df2 = df[df.age > 3]
df2.createOrReplaceTempView("people")

In [ ]:
df3 = spark.sql("select * from people")
sorted(df2.collect()) == sorted(df3.collect())

In [ ]:
spark.catalog.dropTempView("people")

In [ ]:
# createTempView(self, name: str) -> None
#  |      Creates a local temporary view with this :class:`DataFrame`.
#  |      
#  |      The lifetime of this temporary table is tied to the :class:`SparkSession`
#  |      that was used to create this :class:`DataFrame`.
#  |      throws :class:`TempTableAlreadyExistsException`, if the view name already exists in the
#  |      catalog.

In [ ]:
# crossJoin(self, other: 'DataFrame') -> 'DataFrame'
#  |      Returns the cartesian product with another :class:`DataFrame`.

from pyspark.sql import Row
df = spark.createDataFrame([(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
df2 = spark.createDataFrame([Row(name="Tom", height=198), Row(name="Bob", height=175)])

# df.crossJoin(df2.name).show()
df.crossJoin(df2).show()

In [ ]:
# withColumn - very useful function in real life. Allows you to add a new column, update an existing col value, change the datatype of a col

df = spark.createDataFrame(data = [("Shishir", 25, "Male"), ("Rahul", 27, "Male")], schema = ["Name", "Age", "Gender"])
df.show()
df.printSchema()
# By default, spark treats a number as long. Say, we need to change the datatype of age to int

In [ ]:
# withColumn(colName: str, col: pyspark.sql.column.Column) -> 'DataFrame' 

# Returns a new :class:`DataFrame` by adding a column or replacing the
#     existing column that has the same name.

# The column expression must be an expression over this :class:`DataFrame`; attempting to add
#     a column from some other :class:`DataFrame` will raise an error.

from pyspark.sql.functions import col

# colName is caseInsesitive - meaning if the col exists (doesn't matter if the name is in uppercase or lowercase, spark is going to modify the existing column)
# Original df had 'Age' column, but within withColumn function we passed 'age'. Spark identifies and knows that we want to modify the exisitng 'Age' column. 
# df.withColumn(colName, col)-- 2nd parameter is a Column instance always
df2 = df.withColumn('age', col('age').cast('Integer'))
df2.printSchema()
df2.show()

In [ ]:
# col(col: str) -> pyspark.sql.column.Column
#     Returns a :class:`~pyspark.sql.Column` based on the given column name.

# lit(col: Any) -> pyspark.sql.column.Column
#     Creates a :class:`~pyspark.sql.Column` of literal value.

# If you want to provide hard-coded value as col instance, use lit() function
            
from pyspark.sql.functions import lit
    
# Introduce a new column -- country col doesn't exist. 
df3 = df2.withColumn('country', lit("India"))
df3.show()

In [ ]:
# introduce a new column, using values from another col in df
df3.withColumn('doubleAge', df2.age * 2).show()
df3.withColumn('doubleAge', col("age") * 2).show()

# There are 2 ways to pick a column 
# 1. df.colName  --> returns column instance
# 2. using col() function -- col(colName) --> returns column instance

In [ ]:
df3.age

In [ ]:
col('age')

In [ ]:
# Imp NOTE: Pyspark DataFrames are immutable. Whatever transformation you apply, it's going to return you a new DataFrame. 
# Changes won't be performed in the existing dataframe

In [ ]:
# withColumnRenamed(existing: str, new: str) -> 'DataFrame' method of pyspark.sql.dataframe.DataFrame instance
#     Returns a new :class:`DataFrame` by renaming an existing column.
#     This is a no-op if the schema doesn't contain the given column name.

# df3.withColumn(colName: str, col: pyspark.sql.column.Column) -- was used to change/maniupulate column values or to introduce new column
# df3.withColumnRenamed(oldColName: str, newColName: str) -- used for renaming an existing column. If the column doesn't exist, no operation is performed

df4 = df3.withColumnRenamed('age', 'age2')

df3.show()

df4.show()

#NOTE: It's clearly evident that the transformation didn't change the existing df. Instead it returned a new dataFrame with the updated colName.

help(df3.withColumnRenamed)

In [ ]:
# class StructType(DataType)
# Struct type, consisting of a list of :class:`StructField`.
#  |  
#  |  This is the data type representing a :class:`Row`.
#  |  
#  |  Iterating a :class:`StructType` will iterate over its :class:`StructField`\s.
#  |  A contained :class:`StructField` can be accessed by its name or position.
        
from pyspark.sql.types import *

struct1 = StructType([StructField("f1", StringType())])

struct2 = StructType().add(StructField("f1", StringType()))

struct1 == struct2

In [ ]:
data = [("Alice", ["Java", "Scala"]), ("Bob", ["Java", "Scala"])]

schema = StructType([\
               StructField("name", StringType()), \
               StructField("languageSkills", ArrayType(StringType()))
           ])
                     
df = spark.createDataFrame(data = data, schema = schema)
df.show()
df.printSchema()

In [ ]:
# StructType, StructField, IntegerType, StringType, ArrayType etc are different classes in pyspark.sql.types module
# StructType class has an add() method which accepts a StructField or colName:str, DataType, Nullability:bool.
# These are just different types that pyspark supports for creating StructType. All are conceptually same.
# Interesting thing to NOTE: DataType can be another StructType as well. This allows us to define complex structure types.

data = [(1, ("Shishir", "Singh"), 25), (2, ("Rahul", "Patil"), 27)]


structName = StructType([StructField("firstName", StringType()), \
                         StructField("lastName", StringType()),])

schema = StructType([\
               StructField("id", IntegerType()), \
               StructField("name", structName), \
               StructField("age", IntegerType())
           ])

schema2 = StructType().add(StructField("id", IntegerType())) \
                      .add(StructField("name", structName)) \
                      .add(StructField("age", IntegerType()))

schema3 = StructType().add("id", IntegerType()) \
                      .add("name", structName) \
                      .add("age", IntegerType())

df = spark.createDataFrame(data = data, schema = schema3)

print(schema == schema2 == schema3)

df.show()

display(df)

df.printSchema()


In [ ]:
# class ArrayType in module pyspark.sql.types:
# ArrayType(elementType: pyspark.sql.types.DataType, containsNull: bool = True)
# The array can contain null (None) values by default

from pyspark.sql.types import * 

# AssertionError: elementType <class 'pyspark.sql.types.StringType'> should be an instance 
#     of <class 'pyspark.sql.types.DataType'>

# ArrayType(StringType) -- X elementType should be an instance of DataType. 
# That is why it's important to add () to StringType() bcoz we need to pass instance of DataType()

ArrayType(StringType()) # by default, array can null 

In [ ]:
ArrayType(StringType(), False) == ArrayType(StringType())

In [ ]:
data = [("abc", [1,2]), ("def", [3,4])]
df = spark.createDataFrame(data, schema = ["id", "numbers"])
df.show()

# spark automatically inferred the datatype by scanning the data. Saw array containing numbers. 
# By default, spark stores numbers in long.
df.printSchema()

# explicility defining schema
schema = StructType().add(StructField("id", StringType())) \
                     .add(StructField("numbers", ArrayType(IntegerType())))

df = spark.createDataFrame(data, schema = schema)
df.show()

# Now we have defined our schema - and array contains integer's
df.printSchema()


In [ ]:
data = [(1,2), (3,4)]

# passes schema as DDL formatted string
df = spark.createDataFrame(data = data, schema = "num1 int, num2 int")
df.show()
df.printSchema()

# say we want to add a new column
# syntax: withColName(colName: str, col: pyspark.sql.column.Column)
# array() -> pyspark.sql.column.Column
# Parameters
#     ----------
#     cols : :class:`~pyspark.sql.Column` or str
#         column names or :class:`~pyspark.sql.Column`\s that have
#         the same data type.

# you can access the array values using positions
df.withColumn('numbers', array(col("num1"), col("num2"))) \
  .withColumn("firstVal", col("numbers")[0]).show()


In [ ]:
from pyspark.sql.functions import *
help(array)

In [ ]:
# some commonly used functions with array in pyspark

# explode(col: 'ColumnOrName') -> pyspark.sql.column.Column
#     Returns a new row for each element in the given array or map.
#     Uses the default column name `col` for elements in the array and
#     `key` and `value` for elements in the map unless specified otherwise.

#  Parameters
#     ----------
#     col : :class:`~pyspark.sql.Column` or str
#         target column to work on.
    
    
#     Returns
#     -------
#     :class:`~pyspark.sql.Column`
#         one row per array item or map key value.

from pyspark.sql import * 

df = spark.createDataFrame([Row(a=1, intlist=[1,2,3], mapfield={"a": "b", "c": "d"})])
df.show()

# added a new column
df.withColumn('num', explode(col('intlist'))).show()

# selecting just the explode column
df.select(explode(col('intlist'))).show()

df.select(explode(col('intlist')).alias('intNum')).show()

df.select(explode(col('mapfield'))).show()

In [ ]:
# split(str: 'ColumnOrName', pattern: str, limit: int = -1) -> pyspark.sql.column.Column
#     Splits str around matches of the given pattern.
    
#     Parameters
#     ----------
#     str : :class:`~pyspark.sql.Column` or str
#         a string expression to split
#     pattern : str
#         a string representing a regular expression. The regex string should be
#         a Java regular expression.
#     limit : int, optional
#         an integer which controls the number of times `pattern` is applied.
    
#         * ``limit > 0``: The resulting array's length will not be more than `limit`, and the
#                          resulting array's last entry will contain all input beyond the last
#                          matched pattern.
#         * ``limit <= 0``: `pattern` will be applied as many times as possible, and the resulting
#                           array can be of any size.
    
#         .. versionchanged:: 3.0
#            `split` now takes an optional `limit` field. If not provided, default limit value is -1.
    
    
# split(ColumnOrName) -- splits the string around the delimiter and returns an array
data = [(1, "Shishir", ".net, java, aws"), (1, "Rahul", "c++, python, golang")]
df = spark.createDataFrame(data = data, schema = ["id", "name", "skills"])
df.show()
df.printSchema()

df2 = df.withColumn('skillsArray', split(col('skills'), ','))
df2.show()
df2.printSchema()

In [ ]:
# array(*cols: Union[ForwardRef('ColumnOrName'), List[ForwardRef('ColumnOrName_')], Tuple[ForwardRef('ColumnOrName_'), ...]]) -> pyspark.sql.column.Column
#     Creates a new array column.

#  Parameters
#     ----------
#     cols : :class:`~pyspark.sql.Column` or str
#         column names or :class:`~pyspark.sql.Column`\s that have
#         the same data type.
    
#     Returns
#     -------
#     :class:`~pyspark.sql.Column`
#         a column of array type.

df = spark.createDataFrame([("Alice", 2), ("Bob", 5)], ("name", "age"))
df.show()

df.select(array(col('age'), col('age')).alias('numbers')).show()

df.select(array('age', 'age').alias('numbers')).show()

df = spark.createDataFrame(data = [("Alice", "java", "aws"), ("Bob", "python", 'gcp')], 
                        schema = ["name", "primarySkill", "secondarySkill"])

df.show()

df.withColumn('skills', array(col('primarySkill'), col('secondarySkill'))).show()

In [ ]:
df = spark.createDataFrame(data = [("Alice", "java", "aws"), ("Bob", "python", 'gcp')], 
                        schema = ["name", "primarySkill", "secondarySkill"])

df.show()

df2 = df.withColumn('skills', array(col('primarySkill'), col('secondarySkill')))

df2.show()

# this is case sensitive. 

# notice: how the output changes when I change the casing of 'java' string
print('Does it contain JAVA?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'JAVA')).show()

print('Does it contain Java?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'Java')).show()

print('Does it contain java?')
df2.withColumn('hasJavaSkill', array_contains(col('skills'), 'java')).show()

In [ ]:
# Help on class MapType in module pyspark.sql.types:

# class MapType(DataType)
#  |  MapType(keyType: pyspark.sql.types.DataType, valueType: pyspark.sql.types.DataType, valueContainsNull: bool = True)
#  |  
#  |  Map data type.
#  |  
#  |  Parameters
#  |  ----------
#  |  keyType : :class:`DataType`
#  |      :class:`DataType` of the keys in the map.
#  |  valueType : :class:`DataType`
#  |      :class:`DataType` of the values in the map.
#  |  valueContainsNull : bool, optional
#  |      indicates whether values can contain null (None) values.
#  |  
#  |  Notes
#  |  -----
#  |  Keys in a map data type are not allowed to be null (None).

from pyspark.sql.types import *
from pyspark.sql.functions import *

data = [("Shishir", {"eye": "black", "height": 180}), ("Rahul", {"eye": "brown", "height": 175})]

# we have not specified the schema explicitly. Spark infers the schema & Sees properties is of map type
# where key and value are both string. Although we passed height as int, it treats it as string
df = spark.createDataFrame(data = data, schema = ["name", "properties"])
df.show()
df.printSchema()

# defining schema explicitly -- name of string and properties of map type

# MapType(keyType: pyspark.sql.types.DataType, valueType: pyspark.sql.types.DataType, valueContainsNull: bool = True)
# Parameters
#  |  ----------
#  |  keyType : :class:`DataType`
#  |      :class:`DataType` of the keys in the map.
#  |  valueType : :class:`DataType`
#  |      :class:`DataType` of the values in the map.
#  |  valueContainsNull : bool, optional
#  |      indicates whether values can contain null (None) values.
#  |  
#  |  Notes
#  |  -----
#  |  Keys in a map data type are not allowed to be null (None).

schema = StructType().add(StructField("name", StringType())) \
                     .add(StructField("properties", MapType(StringType(), StringType())))

df = spark.createDataFrame(data = data, schema = schema)
df.show(truncate=False)
df.printSchema()


# accessing map's key values

# df.properties or col('properties') both will return map column. To access a key's value - use mp[keyName]
# col('properties') is the map column which I want to access and then the key
df = df.withColumn('eyeColor', col('properties')['eye']) \
  .withColumn('height', col('properties')['height']) 

df.show(truncate=False)

In [ ]:
# explode(col: 'ColumnOrName') -> pyspark.sql.column.Column

# explode() on map column basically produces 2 columns for key and value. Works the same as in case of array, just that in arrays it produces 1 column
df.select(df.name, df.properties, df.eyeColor, df.height, explode(col("properties")).alias("key", "value")) \
.show(truncate=False)

In [ ]:
df.show(truncate=False)

# map_keys(col: 'ColumnOrName') -> pyspark.sql.column.Column
#     Collection function: Returns an unordered array containing the keys of the map.


        
# added 2 new columns extracting keys and values using map_keys() and map_values() functions.
# map_keys() and map_values() both return pyspark.sql.column.Column instance of array type
df_new = df.withColumn('mapKeys', map_keys(col('properties'))) \
 .withColumn('mapValues', map_values(col('properties'))) 

df_new.show()
df_new.printSchema()


In [ ]:
# Below line imports all public names (classes, functions, constants) defined in the pyspark.sql.types module 
# into our current namespace
from pyspark.sql.types import *

# this is how you create a complex schema
schema = StructType().add(StructField("name", StringType())) \
            .add(StructField("properties", MapType(StringType(), StringType()))) \
            .add(StructField("eyeColor", StringType())) \
            .add(StructField("height", StringType())) \
            .add(StructField("mapKeys", ArrayType(StringType()))) \
            .add(StructField("mapValues", ArrayType(StringType()))) 

# pyspark is smart enough & perfectly capable to determine that it's a StructField.
# same schema defined without using StructField() -- by just specifying the colName, DataType(), nullability[optional]
schema = StructType().add("name", StringType()) \
            .add("properties", MapType(StringType(), StringType())) \
            .add("eyeColor", StringType()) \
            .add("height", StringType()) \
            .add("mapKeys", ArrayType(StringType())) \
            .add("mapValues", ArrayType(StringType()))
                 
schema
                 

In [ ]:
# Commonly used Pyspark Modules

# Modules                   Purpose                     Example
# pyspark.sql           	Core SQL/DataFrame API     	SparkSession, DataFrame
# pyspark.sql.functions	    Built-in column functions 	col, when, lit, sum, avg, corr
# pyspark.sql.types	        Data types & schemas      	StringType, StructType
# pyspark.sql.window    	Window functions         	Window.partitionBy
# pyspark.ml                Machine learning            Pipeline, VectorAssembler



# Hierarchy is something like this

# Package
#  └── Module
#       ├── Classes        → define objects with state + behavior
#       ├── Functions      → standalone reusable logic
#       ├── Constants      → fixed values / configuration
#       ├── Variables      → runtime data
#       └── Imports        → things pulled from other modules

        
# Example: pyspark.sql.functions (a module)
# pyspark.sql.functions   ← module
#  ├── col()              ← function
#  ├── avg()              ← function
#  ├── when()             ← function
#  ├── lit()              ← function
#  └── ...
    
    
# pyspark.sql.types       ← module
#  ├── StringType         ← class
#  ├── IntegerType        ← class
#  ├── StructType         ← class
#  ├── StructField        ← class
#  └── ...

# Key clarifications

# 1. A module is just a .py file (or compiled equivalent)
# 2. A package is a folder of modules
# 3. Inside a module you can have:
#     Classes
#     Functions
#     Constants
#     Variables
#     Imports
# 4. Functions do not have to be inside classes in Python
# 5. Classes are for grouping data + behavior; functions are for standalone behavior



# How come a function exists separately inside a module. Generally functions are defined in a class right?
# You’re right that in many OOP designs, functions live inside classes.
# But Python supports multiple styles:

# Object-Oriented

# Functional

# Procedural

# PySpark mixes these styles for usability.

# Example:
    
# from pyspark.sql import functions as F

# df.select(F.col("age"), F.avg("salary"))


# Here: col, avg are just functions, not class methods.
# They return a Column object.
# That Column is later used by DataFrame methods like select, agg.

# So the design is:
# function → returns an object → consumed by class method

# Why Spark chose this design

# 1. SQL-like feel
# select(col("a"), sum("b"))

# Looks similar to SQL:
# SELECT a, SUM(b) FROM table

# 2. Language-agnostic API design

# Spark is written in Scala. Python API mirrors Scala API:

# col("age")
# avg("salary")

In [ ]:
lst = [1,2,3,4]
print(lst)

In [ ]:
from pyspark.sql.types import Row

help(Row)